In [193]:
import importlib
import tests.tokenizer
import os
import json
importlib.reload(tests.tokenizer)
from tests.tokenizer import get_tokenizer

In [194]:
vocab = {0: b' ', 1: b'a', 2:b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at', 1000 : b'&'}
merges = [(b't', b'h'), (b' ', b'c'), (b' ', b'a'), (b'th', b'e'), (b' a',b't')]
tokenizer = get_tokenizer(vocab, merges, ["&"])

In [153]:
encoded = tokenizer.encode("the cat ate the cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat ate") 

In [154]:
decoded = tokenizer.decode(encoded)
decoded

'the cat ate the cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat atethe cat ate'

In [155]:
print(tokenizer.merges_look_up)

{(b't', b'h'): 0, (b' ', b'c'): 1, (b' ', b'a'): 2, (b'th', b'e'): 3, (b' a', b't'): 4}


In [156]:
tokenizer.encode_word(" cat") 

[7, 1, 5]

In [157]:
tokenizer.encode_word("the") 

[9]

In [158]:
tokenizer.encode_word(" ate") 

[10, 3]

In [159]:
word_bytes = "cat".encode("utf-8")
[b for b in word_bytes]

[99, 97, 116]

In [160]:
def get_tokenizer_from_vocab_merges_path(
    vocab_path: str | os.PathLike,
    merges_path: str | os.PathLike,
    special_tokens: list[str] | None = None,
):
    gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
    with open(vocab_path) as vocab_f:
        gpt2_vocab = json.load(vocab_f)
    gpt2_bpe_merges = []
    with open(merges_path) as f:
        for line in f:
            cleaned_line = line.rstrip()
            if cleaned_line and len(cleaned_line.split(" ")) == 2:
                gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
    # The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
    # just return the original bytes, so we don't force students to use
    # any particular encoding scheme.
    vocab = {
        gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
        for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
    }
    # If any of the special tokens don't exist in the vocab, append them to the vocab.
    if special_tokens:
        for special_token in special_tokens:
            byte_encoded_special_token = special_token.encode("utf-8")
            if byte_encoded_special_token not in set(vocab.values()):
                vocab[len(vocab)] = byte_encoded_special_token

    merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_bpe_merges
    ]
    return get_tokenizer(vocab, merges, special_tokens)



In [200]:
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode

VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"

In [201]:
def test_encode_iterable_tinystories_sample_roundtrip():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    all_ids = []
    with open(FIXTURES_PATH / "tinystories_sample.txt") as f:
        for _id in tokenizer.encode_iterable(f):
            all_ids.append(_id)
    with open(FIXTURES_PATH / "tinystories_sample.txt") as f:
        corpus_contents = f.read()
    assert tokenizer.decode(all_ids) == corpus_contents

In [202]:
sorted(["<|endoftext|>", "<|endoftext|><|endoftext|>"]).reverse()

In [203]:
test_encode_iterable_tinystories_sample_roundtrip()

TypeError: 'NoneType' object is not iterable